# Практическое долгосрочное задание 1
## 👄 Интерполяция формы губ по видео



## ЧАСТЬ 0. 🎯 Введение

Нам в руки попали **обрывки данных** — редкие снимки губ человека, произносящего звуки **восточноевропейского алфавита** 🕵️

Наш агент, внедрённый глубоко под прикрытием, снимал его **раз в секунду** — строго по **равномерной временно́й сетке**. Никаких отклонений, никаких дополнительных замеров. Только сухие факты: кадр, ещё кадр, и снова тишина.




Между этими кадрами — **пустота**. Звуки шли один за другим, но мы видим лишь **редкие вспышки мимики**: губы то округляются, то растягиваются, то сжимаются — и всё это **урывками**, без плавных переходов. Мимика кажется **рваной**, движения — **дёргаными**. Смотреть на такое больно 😖

---

### 🚀 Наша финальная задача

#### Научиться воспроизводить **гладкую анимацию** для… 🌀

…впрочем, узнаем **потом**. 😏

---

### 🎬 А пока — начнём с малого!

#### Попробуем восстановить хотя бы **отдельные звуки** 🗣️

<table>
<tr>
<td align="center">👄</td>
<td align="center">🎞️</td>
<td align="center">✨</td>
<td align="center">🔊</td>
</tr>
<tr>
<td align="center">губы</td>
<td align="center">кадры</td>
<td align="center">сгладим</td>
<td align="center">зазвучит!</td>
</tr>
</table>

---

> 💭 **Шаг за шагом:**
> 1. 📍 Соберём редкие кадры
> 2. 🧮 Построим интерполяцию
> 3. 🎥 Оживим анимацию
> 4. 🌍 А потом — уже что-то большее…

In [1]:
#Импорт нужных библиотек

from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import glob
from collections import defaultdict
from draw import *

import pickle

In [2]:
#Загрузка данных
with open('LipsArrayHalfTime.np' , 'rb') as f:
    LipsData = pickle.load(f)

### 📦 Структура данных: `LipsData`

`LipsData` — это **словарь**, ставящий в соответствие каждой букве её временну́ю траекторию движения губ:

$$
\text{LipsData} : \quad \text{буква} \;\longmapsto\; \mathbf{X} \in \mathbb{R}^{n \times 40 \times 2}
$$

---

#### 📐 Что означает каждый индекс

$$
\mathbf{X}[t,\, p,\, c], \qquad
\begin{aligned}
&t = 0, \dots, n-1 \\
&p = 0, \dots, 39 \\
&c \in \{0, 1\}
\end{aligned}
$$

| Индекс | Обозначение | Смысл |
|:---:|:---:|:---|
| $t$ | **время** | номер такта — дискретный отсчёт вдоль оси времени |
| $p$ | **точка** | 40 ключевых точек контура губ (20 внешних + 20 внутренних) |
| $c$ | **координата** | $c=0 \Rightarrow x$, $\;c=1 \Rightarrow y$ на плоскости |

---

#### 🕰️ О такте

> **Такт** — это условная единица времени.  
> Можно считать, что $1$ такт $\approx 0.1$ с, но эта привязка несущественна:  
> для интерполяции важна лишь **последовательность** отсчётов, а не их физическая длительность.

---

#### 🧠 Как это читать

Для фиксированной буквы $\ell$ имеем:

$$
\text{LipsData}[\ell] \;=\; \bigl\{\, \mathbf{x}_t \,\bigr\}_{t=0}^{\,n-1},
\qquad
\mathbf{x}_t \in \mathbb{R}^{40 \times 2}
$$

То есть каждая буква — это **растянутая во времени последовательность** «снимков» формы губ:

$$
\mathbf{x}_0 \;\longrightarrow\; \mathbf{x}_1 \;\longrightarrow\; \cdots \;\longrightarrow\; \mathbf{x}_{n-1}
$$

где каждый $\mathbf{x}_t$ — плоская конфигурация из 40 точек:

$$
\mathbf{x}_t \;=\;
\bigl\{\, (x_p,\, y_p) \,\bigr\}_{p=0}^{39}
$$


### 🎬 Визуализация: функция `animate_lips`

Преподаватель предоставил **готовую функцию** для интерактивной визуализации:

```python
animate_lips(sequence, interval=...)
```
Предлагается оставить значение интервала равным 200. Однако во время интерполяции желательно увеличивать это время пропорционально отнощению желанного количества отрисованных кадров к исходному количеству кадров
### Ниже представлен пример для буквы А

In [4]:
animate_lips(LipsData['А'], interval=200)

## 📐 Часть 1. Интерполяционный многочлен Лагранжа


#### НАПОМНИМ

Пусть даны $n$ узлов $t_0 < t_1 < \dots < t_{n-1}$ и значения функции в них $y_0, y_1, \dots, y_{n-1}$.

**Задача:** найти многочлен $P(t)$ степени $\le n-1$, который **точно проходит через все узлы**:

$$
P(t_i) = y_i, \qquad i = 0, \dots, n-1
$$

---

#### 🧮 Формула Лагранжа

Решение единственно и записывается в виде

$$
P(t) \;=\; \sum_{i=0}^{n-1} y_i \, L_i(t)
$$

где **базисные многочлены Лагранжа**:

$$
L_i(t) \;=\; \prod_{\substack{j=0 \\ j \ne i}}^{n-1} \frac{t - t_j}{\,t_i - t_j\,}
$$

---

#### ✨ Ключевое свойство

$$
L_i(t_k) \;=\;
\begin{cases}
1, & i = k \\[4pt]
0, & i \ne k
\end{cases}
$$

То есть каждый базисный многочлен «отвечает» **только за свой узел**.  
Отсюда мгновенно следует:

$$
P(t_k) = \sum_i y_i \, L_i(t_k) = y_k \quad ✔
$$

---



#### 🤔 Что же делать в части 1 ???

Мы применим эту формулу к **каждой из 80 координат** наших губ:

$$
(x_0, y_0),\;(x_1, y_1),\;\dots,\;(x_{39}, y_{39})
$$

и восстановим **пропущенные кадры** между известными узлами.

#########################################################################################################################

### 🎯 Задание

Дан словарь `LipsData`, где:

- 🔑 ключ — буква,
- 📊 значение — массив `(n, 40, 2)`.

**Требуется:**

1. Для каждой буквы $\ell$ построить **интерполяционный многочлен Лагранжа**
   для каждой из $40 \times 2 = 80$ координат.

   
   В качестве узлов использовать **все исходные кадры**:

   $$
   t_i = 0,\; 1,\; 2,\; \dots,\; n-1
   $$

3. Вычислить значения многочлена на **вдвое более плотной сетке**:

   $$
   t = 0,\; 0.5,\; 1,\; 1.5,\; 2,\; 2.5,\; \dots,\; n-1
   $$

   То есть кадров станет **в 2 раза больше**, чем было изначально.

4. Собрать результат в массив формы `(2n − 1, 40, 2)`.

5. **Визуализировать** результат функцией `animate_lips`.

---

#### ✅ Что должно получиться

- 📈 Каждая буква — восстановленная последовательность `(n, 40, 2)`.
- 🎬 Плавная анимация, где промежуточные кадры заполнены.



In [31]:
##TODO
##ЗДЕСЬ ДОЛЖЕН БЫТЬ ВАШ КОД
## Результат запишите в переменную Task1LispPoints, представляющую собой 
## словарь , где ключ - буква , а значение - массив размера (n, 40, 2) (значение 2n − 1 зависит от буквы)


#SOME CODE
def lagrange_interpolation(t_nodes, y_nodes, t):
    t_nodes = np.asarray(t_nodes, dtype=float)
    y_nodes = np.asarray(y_nodes, dtype=float)              #n - кол-во исходных кадров, m - новых, p - точек, q -координат
    t = np.asarray(t, dtype=float)

    n = len(t_nodes)
    diff = t[:, None] - t_nodes[None, :]                    #O(mn)
    basis = np.empty((len(t), n), dtype=float)

    for i in range(n):                                      
        mask = np.arange(n) != i                           
        numerator = np.prod(diff[:, mask], axis=1)         
        denominator = np.prod(t_nodes[i] - t_nodes[mask])   
        basis[:, i] = numerator / denominator               
                                                            #O(mnn)
    return np.einsum('mi,ipq->mpq', basis, y_nodes)         #O(mnpq)

Task1LispPoints = {}

for letter, data in LipsData.items():
    n = data.shape[0]
    t_nodes = np.arange(n, dtype=float)
    t = np.linspace(0, n - 1, 2*n - 1)

    result = lagrange_interpolation(t_nodes, data, t)
    Task1LispPoints[letter] = result

for letter, data in Task1LispPoints.items():
   print(f"{letter}: {data.shape}")


def cubic_spline(t_nodes, y_nodes, t):
    t_nodes = np.asarray(t_nodes, dtype=float)
    y_nodes = np.asarray(y_nodes, dtype=float)
    t = np.asarray(t, dtype=float)

    n = len(t_nodes)
    h = np.diff(t_nodes)
    c = np.zeros(n, dtype=float)

    if n > 2:
        A = np.zeros((n - 2, n - 2), dtype=float)
        rhs = np.zeros(n - 2, dtype=float)

        for i in range(1, n - 1):
            row = i - 1

            if i > 1:
                A[row, row - 1] = h[i - 1]

            A[row, row] = 2 * (h[i - 1] + h[i])

            if i < n - 2:
                A[row, row + 1] = h[i]

            rhs[row] = 3 * (
                (y_nodes[i + 1] - y_nodes[i]) / h[i]
                - (y_nodes[i] - y_nodes[i - 1]) / h[i - 1]
            )

        c[1:-1] = np.linalg.solve(A, rhs)

    a = y_nodes[:-1]
    b = (
        np.diff(y_nodes) / h
        - h * (2 * c[:-1] + c[1:]) / 3
    )
    d = (
        np.diff(c) / (3 * h)
    )

    result = np.zeros_like(t, dtype=float)
    indices = np.searchsorted(t_nodes, t, side='right') - 1
    indices = np.clip(indices, 0, n - 2)

    dx = t - t_nodes[indices]

    result = (
        a[indices]
        + b[indices] * dx
        + c[indices] * dx**2
        + d[indices] * dx**3
    )

    return result

SplineLipsLetters = {}

for letter, data in LipsData.items():

    n = data.shape[0]
    t_nodes = np.arange(n, dtype=float)
    t = np.linspace(0, n - 1, 2 * n - 1)
    result = np.empty(
        (len(t), 40, 2),
        dtype=float
    )
    for point in range(40):
        result[:, point, 0] = cubic_spline(
            t_nodes,
            data[:, point, 0],
            t
        )
        result[:, point, 1] = cubic_spline(
            t_nodes,
            data[:, point, 1],
            t
        )

    SplineLipsLetters[letter] = result

crop_ranges = {
    'Ё': (6,13),
    'А': (9,19),
    'Б': (13, 18),
    'В': (7,13),
    'Г': (5,10),
    'Д': (5,11),
    'Е': (5,13),
    'Ж': (3,11),
    'З': (5,10),
}
WorkingLips = {}

for letter, (start, end) in crop_ranges.items():
    cropped = SplineLipsLetters[letter][start:end].copy()
    WorkingLips[letter] = cropped

for letter, (start, end) in crop_ranges.items():
    source_data = SplineLipsLetters[letter]
    cropped = source_data[start:end].copy()
    WorkingLips[letter] = cropped
    
    # Выводим информацию о размерах до и после среза
    print(f"{letter}: исходных (сглаженных) = {source_data.shape[0]}, "
          f"срез [{start}:{end}], получено = {cropped.shape[0]}")

word_letters = ['А', 'Б', 'Ё', 'Б', 'А']
combined_parts = [WorkingLips[let] for let in word_letters]
combined_word = np.concatenate(combined_parts, axis=0)

N_comb = combined_word.shape[0]
t_nodes_comb = np.arange(N_comb, dtype=float)
t_comb = np.linspace(0, N_comb - 1, 2 * N_comb - 1)

lagrange_combined = lagrange_interpolation(t_nodes_comb, combined_word, t_comb)

spline_combined = np.empty((len(t_comb), 40, 2), dtype=float)
for p in range(40):
    spline_combined[:, p, 0] = cubic_spline(t_nodes_comb, combined_word[:, p, 0], t_comb)
    spline_combined[:, p, 1] = cubic_spline(t_nodes_comb, combined_word[:, p, 1], t_comb)

"""
временная сложность алгоритма:
O(mnn + mnpq), а в нашем случае m=2n => O(2nnn + nnpq) = O(nn(2n+pq)). Этот алгоритм оказался лучше из-за меньшего кол-ва вложенных циклов.
А также использования einsum(суммирования по Эйнштейну) но этот выигрыш обусловлен тем, что все вычисления перенесены из медленного интерпретатора Python в быстрый компилированный код NumPy.
"""
#Визуализация
animate_lips(lagrange_combined, interval=100)
#animate_lips(spline_combined, interval=100)



Ё: (13, 40, 2)
А: (19, 40, 2)
Б: (23, 40, 2)
В: (15, 40, 2)
Г: (13, 40, 2)
Д: (13, 40, 2)
Е: (13, 40, 2)
Ж: (11, 40, 2)
З: (11, 40, 2)
Ё: исходных (сглаженных) = 13, срез [6:13], получено = 7
А: исходных (сглаженных) = 19, срез [9:19], получено = 10
Б: исходных (сглаженных) = 23, срез [13:18], получено = 5
В: исходных (сглаженных) = 15, срез [7:13], получено = 6
Г: исходных (сглаженных) = 13, срез [5:10], получено = 5
Д: исходных (сглаженных) = 13, срез [5:11], получено = 6
Е: исходных (сглаженных) = 13, срез [5:13], получено = 8
Ж: исходных (сглаженных) = 11, срез [3:11], получено = 8
З: исходных (сглаженных) = 11, срез [5:10], получено = 5


### ⚠️ Внимание!

> **Не ждите, что результаты будут прям красивыми** 😅  


> 🧠 **Если что-то не получается — это нормально!**  
> Подумайте, **почему** результат выглядит именно так,  
> и выпишите свои мысли в ячейке ниже. ✍️

#### ✍️ Запишите здесь, что вы наблюдаете и почему, по вашему мнению,
####    результат выглядит именно так.
\#
    ВАШИ МЫСЛИ
\#

In [ ]:
#########################################################################################################################

## 📐 Часть 2. СКОРО БУДЕТ!!!